## 闭包与装饰器
闭包的概念、无参装饰器、带参装饰器、functools.wraps。（需要函数和 *args/**kwargs 的基础）

### 闭包（Closure）
- 本质：内部函数引用外部函数的变量，并且外部函数返回了内部函数，就构成了闭包
- 三要素：① 函数嵌套；②内层引用外层变量；③外层返回内层函数
- 核心作用：保存状态、延迟执行、封装自由变量
- LEGB作用域：Local（函数内部局部）-> Enclosing （外层闭包） -> Global（模块全局）-> Built-in（内置）
- global 与 nonlocal：改全局变量用 global，改外层闭包变量用 nonlocal



In [ ]:
# ---------- Global ----------
a = 10
def outer_func(x):
    # ---------- Enclosing ----------
    b = 20
    global a
    a = 100
    def inner_func():
        # ---------- Local ----------
        c = 30 # Local
        nonlocal b
        b = 200
        print('x value is', x) # inner 引用 outer 的 x
        print('a value is', a)
        print('b value is', b)
        print('c value is', c)

    return inner_func()

f = outer_func(10)
f

### 装饰器（Decorator）
- 本质：接收函数并返回新函数的可调用对象，通常基于闭包实现
- 三要素：①接收函数参数 ②内部定义 wrapper ③返回 wrapper
- 核心作用：不修改原函数代码，为其增强功能（日志/鉴权/缓存等）
- 关键语法：`@decorator` 语法糖，`functools.wraps` 保留元信息

万能装饰器

In [7]:
from functools import wraps

def log(func):
    @wraps(func)          # 保留原函数 func 的 __name__、__doc__
    def wrapper(*args, **kwargs):
        print(f"==============调用 {func.__name__} 之前==============")

        print(f"func.__name__: {func.__name__}")
        print(f"func.__doc__: {func.__doc__}")

        print(f"位置参数：{args}")
        print(f"关键字参数：{kwargs}")

        result = func(*args, **kwargs)

        print(f"==============调用 {func.__name__} 之后==============")
        print(f"返回值：{result}")
        return result
    return wrapper

@log
def add(a, b):
    """两数相加"""
    return a + b

add(3, 5)

==============调用 add 之前==============
func.__name__: add
func.__doc__: 两数相加
位置参数：(3, 5)
关键字参数：{}
==============调用 add 之后==============
返回值：8


8

计时装饰器

In [12]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        func(*args, **kwargs)
        end = time.time()
        print(f"函数执行时间: {end - start:.6f}秒")
    return wrapper

@timer
def mysum():
    s = 0
    for i in range(1, 1000000):
        s += i

mysum()  # 输出总和和函数执行时间

函数执行时间: 0.034583秒


带参数的装饰器

In [11]:
from functools import wraps

def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                print(f"第 {i + 1} 次调用")
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def say_hi():
    print("hi")

say_hi()

第 1 次调用
hi
第 2 次调用
hi
第 3 次调用
hi
